# Digital Twin — Battery SOC Estimation (Colab Training)

Three training regimes for the same LSTM architecture:
1. **Vanilla LSTM** — MSE loss + dropout (p = 0.05)
2. **Physics-Informed LSTM** — MSE + physics loss + dropout (p = 0.05)
3. **Physics-Informed LSTM (no dropout)** — MSE + physics loss, physics-only regularization

This notebook:
1. Clones the project repo and sets up the environment
2. Loads data and creates DataLoaders
3. Trains all three models sequentially
4. Compares training curves across models

Trained models and per-epoch CSV stats are saved to `trained_models/`.

## 1. Setup — Clone Repo & Check GPU

In [ ]:
import os
import sys

REPO_URL = "https://github.com/juanaponterojas01/Digital_Twin_Battery_with_LSTM.git"
REPO_DIR = "/content/Digital_Twin_Battery_with_LSTM"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repository already exists at {REPO_DIR}")

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q torch torchvision scikit-learn

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Imports & Configuration

In [1]:
import importlib
import os
import torch

import config
importlib.reload(config)

from dataset import create_dataloaders
from engine import train_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config.DEVICE = str(device)
trained_dir = os.path.join(config.PROJECT_DIR, "trained_models")
print(f"Using device: {device}")
print(f"Save directory: {trained_dir}")

Using device: cuda
Save directory: c:\Users\Jaun\Desktop\dt_battery_lstm\trained_models


## 3. Create DataLoaders

In [2]:
train_loader, val_loader, test_loaders = create_dataloaders()

Train: LA92 — 140,675 windows
Val:   US06 — 240 windows
Test:  Cycle1 — 548 windows
Test:  Cycle2 — 556 windows
Test:  Cycle3 — 512 windows
Test:  Cycle4 — 604 windows
Test:  HWFTa — 379 windows
Test:  HWFTb — 379 windows
Test:  UDDS — 1,120 windows
Test:  NN — 584 windows


## 4. Train Model 1 — Vanilla LSTM (MSE Loss + Dropout)

In [3]:
config.DROPOUT_CONDITION = True  # Dropout active during training
config.DROPOUT = 0.00

model1_vanilla, results1_vanilla = train_model(
    is_physics=False,
    beta=0.0,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loaders=test_loaders,
)


Training: vanilla_lstm
Device: cuda
Physics: False  Beta: 0.0  Dropout: 0.0

Model parameters: 17,473



KeyboardInterrupt: 

## 5. Train Model 2 — Physics-Informed LSTM (MSE + Physics Loss + Dropout)

In [ ]:
config.DROPOUT_CONDITION = True  # Dropout active during training and validation
config.DROPOUT = 0.05

model2_physics, results2_physics = train_model(
    is_physics=True,
    beta=config.BETA,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loaders=test_loaders,
)

## 6. Train Model 3 — Physics-Informed LSTM (MSE + Physics Loss, No Dropout)

Dropout is completely disabled (`--no-dropout` regime). The physics consistency loss acts as the sole regularizer. MC uncertainty estimation is **not available** for this model.

In [ ]:
config.DROPOUT_CONDITION = False  # Dropout completely disabled
config.DROPOUT = 0.05  # Still stored in checkpoint metadata, but p=0.0 is used internally

model3_no_dropout, results3_no_dropout = train_model(
    is_physics=True,
    beta=config.BETA,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loaders=test_loaders,
)

## 7. Training Curves Visualization

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

csv_paths = {
    "Vanilla LSTM (dropout)": os.path.join(trained_dir, "training_stats_vanilla_lstm.csv"),
    "Physics LSTM (dropout)": os.path.join(trained_dir, "training_stats_physics_lstm.csv"),
    "Physics LSTM (no dropout)": os.path.join(trained_dir, "training_stats_physics_lstm_no_dropout.csv"),
}

for idx, (title, csv_path) in enumerate(csv_paths.items()):
    if os.path.isfile(csv_path):
        df = pd.read_csv(csv_path)
        axes[idx].plot(df["epoch"], [float(v) for v in df["train_loss"]], label="Train")
        axes[idx].plot(df["epoch"], [float(v) for v in df["val_loss"]], label="Val")
    axes[idx].set_title(f"{title} — Loss")
    axes[idx].set_xlabel("Epoch")
    axes[idx].set_ylabel("Loss")
    axes[idx].legend()
    axes[idx].grid(True)
    axes[idx].set_yscale("log")

plt.tight_layout()
plt.show()

In [ ]:
# Validation RMSE comparison across all models
fig, ax = plt.subplots(figsize=(10, 5))

colors = {"Vanilla LSTM (dropout)": "tab:blue",
          "Physics LSTM (dropout)": "tab:orange",
          "Physics LSTM (no dropout)": "tab:green"}

for label, csv_path in csv_paths.items():
    if os.path.isfile(csv_path):
        df = pd.read_csv(csv_path)
        ax.plot(df["epoch"], [float(v) for v in df["val_rmse"]],
                label=label, color=colors[label])

ax.set_xlabel("Epoch")
ax.set_ylabel("Validation RMSE")
ax.set_title("Validation RMSE Comparison")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## 8. Download Training Results

In [ ]:
!zip -r /content/trained_models.zip /content/Digital_Twin_Battery_with_LSTM/trained_models/

In [ ]:
from google.colab import files
files.download("/content/trained_models.zip")